# Aiscern — Image Deepfake Detector Fine-tuning

**Model**: `google/vit-base-patch16-224` + LoRA

**Run on Kaggle** (30h free GPU/week)

**Output**: `saghi776/aiscern-image-detector`

**Target**: 98%+ accuracy (up from 97%)

In [ ]:
!pip install -q transformers datasets peft accelerate evaluate scikit-learn Pillow huggingface_hub

In [ ]:
HF_TOKEN     = 'YOUR_HF_TOKEN_HERE'
BASE_MODEL   = 'google/vit-base-patch16-224'
PUSH_TO_REPO = 'saghi776/aiscern-image-detector'
MAX_SAMPLES  = 39000  # ALL available (78,000 ÷ 2 per class)
BATCH_SIZE   = 32
EPOCHS       = 10    # Kaggle P100 — image only takes 2.3h total
LR           = 2e-4
import os; os.environ['HF_TOKEN'] = HF_TOKEN
print('Config ✅')

In [ ]:
from datasets import load_dataset, concatenate_datasets
import numpy as np

# Best labeled image deepfake dataset — CIFAKE has clean labels
print('Loading CIFAKE (AI vs Real)...')
cifake = load_dataset(
    'jlbaker361/cifake-real-and-ai-generated-small-images',
    split='train', token=HF_TOKEN
)
label_map = {'FAKE': 1, 'REAL': 0, 'fake': 1, 'real': 0, '1': 1, '0': 0}
cifake = cifake.map(lambda x: {'label': label_map.get(str(x['label']), -1)})
cifake = cifake.filter(lambda x: x['label'] != -1)

print('Loading FaceForensics deepfake faces...')
ff = load_dataset(
    'marcelomoreno26/deepfake-detection',
    split='train', token=HF_TOKEN
)
ff = ff.map(lambda x: {'label': label_map.get(str(x.get('label', x.get('Label', ''))), -1)})
ff = ff.filter(lambda x: x['label'] != -1)

# Select image column
def get_img_col(ds):
    for col in ['image', 'img', 'Image', 'pixel_values']:
        if col in ds.column_names: return col
    return None

# Combine + balance
all_ds = []
for ds in [cifake, ff]:
    img_col = get_img_col(ds)
    if img_col:
        if img_col != 'image':
            ds = ds.rename_column(img_col, 'image')
        all_ds.append(ds.select_columns(['image', 'label']))

combined = concatenate_datasets(all_ds)
real = combined.filter(lambda x: x['label']==0).shuffle(42).select(range(min(MAX_SAMPLES, combined.filter(lambda x: x['label']==0).num_rows)))
fake = combined.filter(lambda x: x['label']==1).shuffle(42).select(range(min(MAX_SAMPLES, combined.filter(lambda x: x['label']==1).num_rows)))
n = min(len(real), len(fake))
balanced = concatenate_datasets([real.select(range(n)), fake.select(range(n))]).shuffle(42)
split = balanced.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(f'Train: {len(train_ds):,}  Eval: {len(eval_ds):,}')

In [ ]:
from transformers import ViTImageProcessor

processor = ViTImageProcessor.from_pretrained(BASE_MODEL)

def preprocess(batch):
    imgs = []
    for img in batch['image']:
        if hasattr(img, 'convert'):
            imgs.append(img.convert('RGB'))
        else:
            from PIL import Image
            import numpy as np
            imgs.append(Image.fromarray(np.uint8(img)).convert('RGB'))
    inputs = processor(images=imgs, return_tensors='np')
    batch['pixel_values'] = inputs['pixel_values']
    return batch

train_ds = train_ds.map(preprocess, batched=True, batch_size=64, remove_columns=['image'])
eval_ds  = eval_ds.map(preprocess,  batched=True, batch_size=64, remove_columns=['image'])
train_ds.set_format('torch'); eval_ds.set_format('torch')
print('Features extracted ✅')

In [ ]:
from transformers import ViTForImageClassification
from peft import LoraConfig, get_peft_model, TaskType
import torch

model = ViTForImageClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: 'real', 1: 'fake'},
    label2id={'real': 0, 'fake': 1},
    ignore_mismatched_sizes=True,
)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.1,
    bias='none',
    target_modules=['query', 'value', 'key', 'dense'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f'Model on {device} ✅')

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate, numpy as np

accuracy = evaluate.load('accuracy')
f1 = evaluate.load('f1')

def compute_metrics(ep):
    preds = np.argmax(ep.predictions, axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=preds, references=ep.label_ids)['accuracy'],
        'f1': f1.compute(predictions=preds, references=ep.label_ids, average='binary')['f1'],
    }

training_args = TrainingArguments(
    output_dir='./image-detector-checkpoints',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    push_to_hub=True,
    hub_model_id=PUSH_TO_REPO,
    hub_token=HF_TOKEN,
    fp16=torch.cuda.is_available(),
    report_to='none',
    logging_steps=50,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)
trainer.train()

results = trainer.evaluate()
print(f"Accuracy: {results['eval_accuracy']:.4f}  F1: {results['eval_f1']:.4f}")
trainer.push_to_hub(commit_message='Fine-tuned ViT image deepfake detector')
processor.push_to_hub(PUSH_TO_REPO, token=HF_TOKEN)
print(f'✅ https://huggingface.co/{PUSH_TO_REPO}')